<a href="https://colab.research.google.com/github/prateeekglitch/aws-serverless-churn-predictor/blob/main/churn_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# install required cloud and ml libraries
!pip install awswrangler scikit-learn pandas > /dev/null 2>&1

import pandas as pd
import awswrangler as wr
import boto3
import numpy as np
import datetime as dt
from google.colab import userdata

# initialize secure aws session via colab secrets
my_session = boto3.Session(
    region_name="ap-south-1",
    aws_access_key_id=userdata.get('AWS_ACCESS_KEY'),
    aws_secret_access_key=userdata.get('AWS_SECRET_KEY')
)

# execute serverless query via athena
sql_query = """
    SELECT CustomerID, InvoiceDate, InvoiceNo, (Quantity * UnitPrice) as Total_Spend
    FROM ecomm_transactions
    WHERE Quantity > 0 AND UnitPrice > 0 AND CustomerID IS NOT NULL
"""

print("querying aws athena...")
df = wr.athena.read_sql_query(sql=sql_query, database="default", boto3_session=my_session)
print(f"extraction complete. shape: {df.shape}")

querying aws athena...
extraction complete. shape: (525386, 4)


In [ ]:
# enforce schema mapping to fix athena lowercase conversion
df.columns = ['CustomerID', 'InvoiceDate', 'InvoiceNo', 'Total_Spend']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# isolate 90-day holdout set to prevent data leakage during rfm calculation
cutoff_date = df['InvoiceDate'].max() - dt.timedelta(days=90)
behavior_df = df[df['InvoiceDate'] < cutoff_date]
future_df = df[df['InvoiceDate'] >= cutoff_date]

# engineer rfm (recency, frequency, monetary) features
NOW = cutoff_date
rfm = behavior_df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (NOW - x.max()).days,
    'InvoiceNo': 'nunique',
    'Total_Spend': 'sum'
}).rename(columns={'InvoiceDate': 'Recency', 'InvoiceNo': 'Frequency', 'Total_Spend': 'Monetary'})

# define binary target variable (churn = 1 if absent in future_df)
future_buyers = future_df['CustomerID'].unique()
rfm['Churn'] = np.where(rfm.index.isin(future_buyers), 0, 1)

print(f"rfm matrix engineered. unique customers: {rfm.shape[0]}")

rfm matrix engineered. unique customers: 3369


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# define feature matrix and target vector
X = rfm[['Recency', 'Frequency', 'Monetary']]
y = rfm['Churn']

# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# initialize and train logistic regression with synthetic minority class balancing
model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

# generate predictions and evaluate performance
predictions = model.predict(X_test)

print("\n--- model performance metrics ---")
print(classification_report(y_test, predictions))


--- model performance metrics ---
              precision    recall  f1-score   support

           0       0.81      0.57      0.67       412
           1       0.54      0.79      0.64       262

    accuracy                           0.66       674
   macro avg       0.68      0.68      0.66       674
weighted avg       0.71      0.66      0.66       674

